In [ ]:
thresholds_test = np.arange(0.1, 0.9, 0.05)
for t in thresholds_test:
    preds = (y_proba_v2 >= t).astype(int)
    r = recall_score(y_val, preds)
    p = precision_score(y_val, preds)
    f = f1_score(y_val, preds)
    print(f"threshold={t:.2f} → Recall={r:.4f} Precision={p:.4f} F1={f:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

thresholds_to_plot = [0.10, 0.25, 0.35, 0.50]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, t in zip(axes, thresholds_to_plot):
    preds = (y_proba_v2 >= t).astype(int)
    cm = confusion_matrix(y_val, preds)
    tn, fp, fn, tp = cm.ravel()

    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Não Churn", "Churn"])
    ax.set_yticklabels(["Não Churn", "Churn"])
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")

    recall = tp / (tp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    ax.set_title(
        f"Threshold = {t:.2f}\n"
        f"Recall={recall:.2f} | Precision={precision:.2f} | F1={f1:.2f}",
        fontsize=10
    )

    for i, j, val in [(0,0,tn),(0,1,fp),(1,0,fn),(1,1,tp)]:
        ax.text(j, i, str(val), ha="center", va="center",
                color="white" if val > cm.max()/2 else "black",
                fontsize=14, fontweight="bold")

plt.suptitle("Comparativo de Matrizes de Confusão — ChurnMLP v2", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("../docs/figures/confusion_matrix_thresholds.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import pandas as pd

COST_FN = 500.0
COST_FP = 50.0
baseline_cost = y_val.sum() * COST_FN

rows = []
for t in np.arange(0.10, 0.85, 0.05):
    tn, fp, fn, tp = confusion_matrix(y_val, (y_proba_v2 >= t).astype(int)).ravel()
    savings = baseline_cost - (fn * COST_FN) - (fp * COST_FP)
    rows.append({"threshold": round(t, 2), "recall": round(tp/(tp+fn), 4),
                 "f1": round(2*tp/(2*tp+fp+fn), 4), "savings": round(savings, 2)})

df_fin = pd.DataFrame(rows)
print(df_fin.to_string(index=False))

# Gráfico
plt.figure(figsize=(8, 4))
plt.plot(df_fin["threshold"], df_fin["savings"], marker="o", color="steelblue", lw=2)
plt.axvline(x=df_fin.loc[df_fin["savings"].idxmax(), "threshold"],
            color="green", linestyle="--",
            label=f"Melhor: {df_fin.loc[df_fin['savings'].idxmax(), 'threshold']}")
plt.xlabel("Threshold")
plt.ylabel("Economia (R$)")
plt.title("Economia por Threshold — ChurnMLP v2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../docs/figures/financial_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# Obter probabilidades da v1
y_proba_v1, _ = get_predictions(model, X_val, device)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (proba, threshold, title) in zip(axes, [
    (y_proba_v1, 0.50, "MLP v1 — threshold=0.50"),
    (y_proba_v2, 0.10, "MLP v2 — threshold=0.35"),
]):
    preds = (proba >= threshold).astype(int)
    cm = confusion_matrix(y_val, preds)
    tn, fp, fn, tp = cm.ravel()

    recall    = tp / (tp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Não Churn", "Churn"])
    ax.set_yticklabels(["Não Churn", "Churn"])
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.set_title(f"{title}\nRecall={recall:.2f} | Precision={precision:.2f} | F1={f1:.2f}")

    for i, j, val in [(0,0,tn),(0,1,fp),(1,0,fn),(1,1,tp)]:
        ax.text(j, i, str(val), ha="center", va="center",
                color="white" if val > cm.max()/2 else "black",
                fontsize=14, fontweight="bold")

plt.suptitle("Comparativo v1 vs v2 — Matriz de Confusão", fontsize=13)
plt.tight_layout()
plt.savefig("../docs/figures/confusion_matrix_v1_v2.png", dpi=150, bbox_inches="tight")
plt.show()